# STGDNet deblurring on real Tianmouc data

This example runs the current Generalized STGDNet model. It follows TMC-Deblur's real-capture preprocessing and blurred/deblurred visualization style, while detecting both 660 μs and 1320 μs AOP modes.

In [ ]:
from pathlib import Path
import os
import cv2
import numpy as np
import torch
from IPython.display import Image, display
from tqdm.auto import tqdm
from tianmoucv.data import TianmoucDataReader

from tianmoucv.proc.deblur import (
    TianmoucDeblurrer, infer_blur_length, make_deblur_comparison,
    prepare_deblur_input, read_exposure_times,
)
from tianmoucv.utils import save_video

In [ ]:
# Point this at one real capture folder whose direct children are cone/ and rod/.
data_path = Path(os.environ.get(
    'TIANMOUC_DATA_PATH',
    '/data51/yihan/tianmoucData/tianmoucReconDataset/test/extreme/shake6',
))
output_dir = Path('deblur_demo_output')
checkpoint = None  # None: download/cache Generalized_Model.pth; local path also works.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
sample_start = 30
sample_count = 70  # Set None to process the complete sequence.
video_fps = 10

if not (data_path / 'cone').is_dir() or not (data_path / 'rod').is_dir():
    raise FileNotFoundError(f'Expected cone/ and rod/ under: {data_path}')
(output_dir / 'deblurred').mkdir(parents=True, exist_ok=True)
(output_dir / 'comparisons').mkdir(exist_ok=True)
print('data:', data_path.resolve())
print('output:', output_dir.resolve())
print('device:', device)

In [ ]:
# Do not enable the newer physical-grid correction here: this reproduces STGDNet training preprocessing.
dataset = TianmoucDataReader(str(data_path), N=1, camera_idx=0, strict=False)
first_sample = dataset[sample_start]
exposure_times = read_exposure_times(data_path)
blur_length, exposure_us, interval_us = infer_blur_length(
    data_path, first_sample, exposure_times=exposure_times
)
deblurrer = TianmoucDeblurrer(checkpoint=checkpoint, device=device)
print(f'samples: {len(dataset)}')
print(f'exposure={exposure_us} us, AOP interval={interval_us} us, blur_length={blur_length}')
print('weights:', deblurrer.checkpoint_path)

In [ ]:
stop = len(dataset) if sample_count is None else min(len(dataset), sample_start + sample_count)
comparison_frames = []

for sample_index in tqdm(range(sample_start, stop), desc='STGDNet'):
    sample = dataset[sample_index]
    # Exposure is indexed by sample['meta']['C_idx'], so auto-exposure captures are supported.
    blur_length, exposure_us, _ = infer_blur_length(
        data_path, sample, exposure_times=exposure_times
    )
    model_input = prepare_deblur_input(sample, blur_length)
    prediction = deblurrer(model_input)[0].cpu()
    prediction_rgb = np.round(prediction.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    comparison = make_deblur_comparison(model_input['frame'], prediction)
    cv2.imwrite(
        str(output_dir / 'deblurred' / f'{sample_index:06d}.png'),
        cv2.cvtColor(prediction_rgb, cv2.COLOR_RGB2BGR),
    )
    cv2.imwrite(
        str(output_dir / 'comparisons' / f'{sample_index:06d}.png'),
        cv2.cvtColor(comparison, cv2.COLOR_RGB2BGR),
    )
    comparison_frames.append(comparison)

video_path = save_video(comparison_frames, output_dir / 'deblur_demo.mp4', fps=video_fps)
preview_path = output_dir / 'comparisons' / f'{stop - 1:06d}.png'
print(f'Saved {stop - sample_start} deblurred frame(s) to {output_dir / "deblurred"}')
print('Saved video:', video_path)
display(Image(filename=str(preview_path)))